# Costs and Taxes Backtest

This notebook handles the implementation-cost layer of the project. It does not estimate HMM or MS-VAR directly. Instead, it expects each model notebook to export monthly target portfolio weights after regime classification and regime-to-asset mapping.

The expected upstream files are:

- outputs/tables/hmm_target_weights.csv
- outputs/tables/msvar_target_weights.csv

Each target-weight file should contain:

Date
regime_label
index_fund
treasury_fund
gold_fund

The three asset-weight columns should sum to 1.0 for each date. The weights should be signal-date target weights. This notebook applies the one-month execution lag internally before calculating portfolio returns, turnover, transaction costs, and tax sensitivity.

## Cost and Tax Treatment

The main cost adjustment is based on monthly rebalancing activity. At each rebalance, the portfolio is compared with the target regime-dependent allocation after allowing weights to drift with asset returns. Total traded notional is measured as the sum of absolute differences between drifted weights and target weights. A transaction cost of 10 basis points is then applied to that traded notional.

The main results are intended to be reported after transaction costs but before taxes. Taxes are handled as sensitivity scenarios rather than as the base-case result because actual tax outcomes depend on investor type, account type, holding period, jurisdiction, and the exact investment vehicle. The tax section below is therefore an approximation, not a full tax-lot accounting model.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
OUTPUT_TABLES_DIR = Path("../outputs/tables")
OUTPUT_FIGURES_DIR = Path("../outputs/figures")

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ASSET_COLS = ["index_fund", "treasury_fund", "gold_fund"]

TRANSACTION_COST_BPS = 10

ASSET_TAX_RATES = {
    "No Tax": {
        "index_fund": 0.00,
        "treasury_fund": 0.00,
        "gold_fund": 0.00,
    },
    "Long-Term Capital Gain Sensitivity": {
        "index_fund": 0.15,
        "treasury_fund": 0.15,
        "gold_fund": 0.15,
    },
    "Gold Collectibles Sensitivity": {
        "index_fund": 0.15,
        "treasury_fund": 0.15,
        "gold_fund": 0.28,
    },
    "High Short-Term Sensitivity": {
        "index_fund": 0.35,
        "treasury_fund": 0.35,
        "gold_fund": 0.35,
    },
}

In [2]:
target_weight_template = pd.DataFrame({
    "Date": ["2000-01-31", "2000-02-29", "2000-03-31"],
    "regime_label": ["Expansion", "Contraction", "Expansion"],
    "index_fund": [0.70, 0.20, 0.70],
    "treasury_fund": [0.20, 0.50, 0.20],
    "gold_fund": [0.10, 0.30, 0.10],
})

template_path = OUTPUT_TABLES_DIR / "target_weights_template.csv"
target_weight_template.to_csv(template_path, index=False)

print(f"Template saved to: {template_path}")
target_weight_template

Template saved to: ..\outputs\tables\target_weights_template.csv


,Date,regime_label,index_fund,treasury_fund,gold_fund
0,2000-01-31,Expansion,0.7,0.2,0.1
1,2000-02-29,Contraction,0.2,0.5,0.3
2,2000-03-31,Expansion,0.7,0.2,0.1


## Required Target-Weight Format

The model notebooks should export target weights after the regime-to-asset mapping step.

For HMM, the expected file is:

outputs/tables/hmm_target_weights.csv

For MS-VAR, the expected file is:

outputs/tables/msvar_target_weights.csv

Required columns:

Date
regime_label
index_fund
treasury_fund
gold_fund

The weights should be signal-date target weights. This notebook applies the execution lag internally. The three asset weights must sum to 1.0 for every row.

In [3]:
asset_returns = pd.read_csv(
    DATA_DIR / "market_clean.csv",
    parse_dates=["Date"]
).set_index("Date")

asset_returns = asset_returns[ASSET_COLS]

asset_returns.head()

,index_fund,treasury_fund,gold_fund
Date,,,
1990-02-28,0.012748,-0.003267,0.017748
1990-03-31,0.027663,-0.003589,-0.051637
1990-04-30,-0.025015,-0.027230,-0.001374
1990-05-31,0.096928,0.044975,-0.013624
1990-06-30,-0.006555,0.022940,-0.003990


In [4]:
model_weight_files = {
    "HMM": OUTPUT_TABLES_DIR / "hmm_target_weights.csv",
    "MSVAR": OUTPUT_TABLES_DIR / "msvar_target_weights.csv",
}

model_weights = {}

for model_name, path in model_weight_files.items():
    if path.exists():
        weights = pd.read_csv(path, parse_dates=["Date"]).set_index("Date")
        model_weights[model_name] = weights
        print(f"Loaded {model_name} target weights from {path}")
    else:
        print(f"Missing {model_name} target weights: {path}")

print(f"\nAvailable model weight files: {len(model_weights)}")

Missing HMM target weights: ..\outputs\tables\hmm_target_weights.csv
Missing MSVAR target weights: ..\outputs\tables\msvar_target_weights.csv

Available model weight files: 0


In [5]:
def validate_target_weights(weights, asset_cols=ASSET_COLS):
    required_cols = ["regime_label"] + asset_cols
    missing_cols = [col for col in required_cols if col not in weights.columns]

    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    if weights[asset_cols].isna().any().any():
        raise ValueError("Target weights contain missing values.")

    weight_sums = weights[asset_cols].sum(axis=1)

    if not np.allclose(weight_sums, 1.0, atol=1e-6):
        bad_rows = weights.loc[~np.isclose(weight_sums, 1.0, atol=1e-6)]
        raise ValueError(
            "Some target-weight rows do not sum to 1. "
            f"Example rows:\n{bad_rows.head()}"
        )

    return True


for model_name, weights in model_weights.items():
    validate_target_weights(weights)
    print(f"{model_name} target weights passed validation.")

In [6]:
if len(model_weights) == 0:
    raise FileNotFoundError(
        "No official model target-weight files were found. "
        "A template has been saved to outputs/tables/target_weights_template.csv. "
        "Please ask the HMM and MS-VAR notebooks to export hmm_target_weights.csv "
        "and/or msvar_target_weights.csv before running the cost and tax analysis."
    )

FileNotFoundError: No official model target-weight files were found. A template has been saved to outputs/tables/target_weights_template.csv. Please ask the HMM and MS-VAR notebooks to export hmm_target_weights.csv and/or msvar_target_weights.csv before running the cost and tax analysis.

In [7]:
def backtest_with_transaction_costs(
    target_weights,
    asset_returns,
    transaction_cost_bps=10,
    apply_signal_lag=True,
    include_initial_trade_cost=False,
):
    weights = target_weights[ASSET_COLS].copy()
    returns = asset_returns[ASSET_COLS].copy()

    if apply_signal_lag:
        weights = weights.shift(1)

    weights = weights.dropna()

    common_index = weights.index.intersection(returns.index)
    weights = weights.loc[common_index]
    returns = returns.loc[common_index]

    records = []
    previous_target = weights.iloc[0]

    for i, date in enumerate(common_index):
        current_target = weights.loc[date]
        current_returns = returns.loc[date]

        if i == 0:
            if include_initial_trade_cost:
                trade_notional = current_target.abs().sum()
            else:
                trade_notional = 0.0
        else:
            previous_returns = returns.loc[common_index[i - 1]]

            drifted_weights = previous_target * (1 + previous_returns)
            drifted_weights = drifted_weights / drifted_weights.sum()

            trade_notional = (current_target - drifted_weights).abs().sum()

        transaction_cost = trade_notional * transaction_cost_bps / 10000

        gross_return = (current_target * current_returns).sum()
        net_return = gross_return - transaction_cost

        records.append({
            "Date": date,
            "gross_return": gross_return,
            "net_return_after_costs": net_return,
            "trade_notional": trade_notional,
            "one_way_turnover": trade_notional / 2,
            "transaction_cost": transaction_cost,
            "index_fund_weight": current_target["index_fund"],
            "treasury_fund_weight": current_target["treasury_fund"],
            "gold_fund_weight": current_target["gold_fund"],
        })

        previous_target = current_target

    return pd.DataFrame(records).set_index("Date")

In [8]:
cost_results = {}

for model_name, weights in model_weights.items():
    result = backtest_with_transaction_costs(
        target_weights=weights,
        asset_returns=asset_returns,
        transaction_cost_bps=TRANSACTION_COST_BPS,
        apply_signal_lag=True,
        include_initial_trade_cost=False,
    )

    cost_results[model_name] = result

    output_path = OUTPUT_TABLES_DIR / f"{model_name.lower()}_monthly_returns_with_costs.csv"
    result.to_csv(output_path)

    print(f"Saved {model_name} monthly returns with costs to {output_path}")

In [9]:
def annualized_return(monthly_returns):
    return (1 + monthly_returns).prod() ** (12 / len(monthly_returns)) - 1


def annualized_volatility(monthly_returns):
    return monthly_returns.std() * np.sqrt(12)


def sharpe_ratio(monthly_returns, risk_free_rate=0.0):
    excess_returns = monthly_returns - risk_free_rate / 12
    return excess_returns.mean() / excess_returns.std() * np.sqrt(12)


def max_drawdown(monthly_returns):
    cumulative = (1 + monthly_returns).cumprod()
    running_max = cumulative.cummax()
    drawdown = cumulative / running_max - 1
    return drawdown.min()


def summarize_performance(monthly_returns):
    return pd.Series({
        "Annualized Return": annualized_return(monthly_returns),
        "Annualized Volatility": annualized_volatility(monthly_returns),
        "Sharpe Ratio": sharpe_ratio(monthly_returns),
        "Max Drawdown": max_drawdown(monthly_returns),
    })

In [10]:
summary_blocks = []

for model_name, result in cost_results.items():
    summary = pd.DataFrame({
        f"{model_name} Gross": summarize_performance(result["gross_return"]),
        f"{model_name} Net After Costs": summarize_performance(result["net_return_after_costs"]),
    })

    summary.loc["Average Trade Notional", f"{model_name} Gross"] = np.nan
    summary.loc["Average Trade Notional", f"{model_name} Net After Costs"] = result["trade_notional"].mean()

    summary.loc["Average One-Way Turnover", f"{model_name} Gross"] = np.nan
    summary.loc["Average One-Way Turnover", f"{model_name} Net After Costs"] = result["one_way_turnover"].mean()

    summary.loc["Average Monthly Transaction Cost", f"{model_name} Gross"] = np.nan
    summary.loc["Average Monthly Transaction Cost", f"{model_name} Net After Costs"] = result["transaction_cost"].mean()

    summary_blocks.append(summary)

transaction_cost_summary = pd.concat(summary_blocks, axis=1)

transaction_cost_summary.to_csv(
    OUTPUT_TABLES_DIR / "transaction_cost_summary.csv"
)

transaction_cost_summary

ValueError: No objects to concatenate

In [ ]:
for model_name, result in cost_results.items():
    cumulative = (1 + result[["gross_return", "net_return_after_costs"]]).cumprod() - 1

    plt.figure(figsize=(12, 6))
    plt.plot(cumulative.index, cumulative["gross_return"], label=f"{model_name} Gross")
    plt.plot(cumulative.index, cumulative["net_return_after_costs"], label=f"{model_name} Net After Costs")

    plt.title(f"{model_name}: Gross vs Net Performance After Transaction Costs")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    output_path = OUTPUT_FIGURES_DIR / f"{model_name.lower()}_gross_vs_net_transaction_costs.png"
    plt.savefig(output_path, dpi=300)
    plt.show()

    print(f"Saved figure to {output_path}")

In [11]:
def apply_asset_level_tax_sensitivity(
    target_weights,
    asset_returns,
    tax_rates,
    apply_signal_lag=True,
):
    weights = target_weights[ASSET_COLS].copy()
    returns = asset_returns[ASSET_COLS].copy()

    if apply_signal_lag:
        weights = weights.shift(1)

    weights = weights.dropna()

    common_index = weights.index.intersection(returns.index)
    weights = weights.loc[common_index]
    returns = returns.loc[common_index]

    tax_rates_series = pd.Series(tax_rates)

    records = []
    previous_target = weights.iloc[0]

    for i, date in enumerate(common_index):
        current_target = weights.loc[date]

        if i == 0:
            sell_weights = pd.Series(0.0, index=ASSET_COLS)
            prior_asset_returns = pd.Series(0.0, index=ASSET_COLS)
        else:
            previous_returns = returns.loc[common_index[i - 1]]

            drifted_weights = previous_target * (1 + previous_returns)
            drifted_weights = drifted_weights / drifted_weights.sum()

            sell_weights = (drifted_weights - current_target).clip(lower=0)
            prior_asset_returns = previous_returns

        taxable_gain_proxy = sell_weights * prior_asset_returns.clip(lower=0)
        tax_drag = (taxable_gain_proxy * tax_rates_series).sum()

        records.append({
            "Date": date,
            "tax_drag": tax_drag,
            "index_fund_sell_weight": sell_weights["index_fund"],
            "treasury_fund_sell_weight": sell_weights["treasury_fund"],
            "gold_fund_sell_weight": sell_weights["gold_fund"],
        })

        previous_target = current_target

    return pd.DataFrame(records).set_index("Date")

In [ ]:
tax_results = {}

for model_name, weights in model_weights.items():
    model_tax_results = {}

    for scenario_name, tax_rates in ASSET_TAX_RATES.items():
        tax_drag = apply_asset_level_tax_sensitivity(
            target_weights=weights,
            asset_returns=asset_returns,
            tax_rates=tax_rates,
            apply_signal_lag=True,
        )

        base_returns = cost_results[model_name]["net_return_after_costs"]
        common_index = base_returns.index.intersection(tax_drag.index)

        after_tax_returns = (
            base_returns.loc[common_index]
            - tax_drag.loc[common_index, "tax_drag"]
        )

        model_tax_results[scenario_name] = after_tax_returns

    tax_results[model_name] = pd.DataFrame(model_tax_results)

    output_path = OUTPUT_TABLES_DIR / f"{model_name.lower()}_tax_sensitivity_returns.csv"
    tax_results[model_name].to_csv(output_path)

    print(f"Saved {model_name} tax sensitivity returns to {output_path}")

In [ ]:
tax_summary_blocks = []

for model_name, returns_df in tax_results.items():
    tax_summary = pd.DataFrame({
        scenario: summarize_performance(returns_df[scenario])
        for scenario in returns_df.columns
    })

    tax_summary.columns = [f"{model_name} - {col}" for col in tax_summary.columns]
    tax_summary_blocks.append(tax_summary)

tax_sensitivity_summary = pd.concat(tax_summary_blocks, axis=1)

tax_sensitivity_summary.to_csv(
    OUTPUT_TABLES_DIR / "tax_sensitivity_summary.csv"
)

tax_sensitivity_summary

In [ ]:
for model_name, returns_df in tax_results.items():
    cumulative = (1 + returns_df).cumprod() - 1

    plt.figure(figsize=(12, 6))

    for col in cumulative.columns:
        plt.plot(cumulative.index, cumulative[col], label=col)

    plt.title(f"{model_name}: Tax Sensitivity Analysis")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    output_path = OUTPUT_FIGURES_DIR / f"{model_name.lower()}_tax_sensitivity.png"
    plt.savefig(output_path, dpi=300)
    plt.show()

    print(f"Saved figure to {output_path}")

## Notes

This notebook is designed to run after the model notebooks export target weights. The cost section measures the performance drag from regime-driven rebalancing. The tax section is included only as a sensitivity analysis, since actual after-tax results would require investor-specific assumptions and full tax-lot accounting.

Once the HMM and MS-VAR target-weight files are available, rerun this notebook from the top to generate the cost-adjusted and tax-sensitivity outputs.